# Masking Actions on Unstructured Data

This notebook demonstrates how to apply **masking actions** to PII entities detected in unstructured text.

The workflow has two steps:
1. **Classify** — detect PII spans in the text using `READIAnalyzer` with the `PII_NO_MODEL` detection type, which relies exclusively on regex-based and dictionary-based identifiers (no ML models required).
2. **Mask** — apply a masking action to each detected entity span using the functions from `risk_assessment.masking.actions`.

The available masking actions are:

| Action | Description |
|---|---|
| `tagging_factory()` | Replaces each unique value with a stable label, e.g. `EMAIL-1`, `EMAIL-2` |
| `tagging_with_hash` | Replaces each value with a type-prefixed hash suffix, e.g. `EMAIL-a3f2c` |
| `redact_factory()` | Replaces any entity with a fixed-width placeholder, e.g. `XXX` |
| `redact_size_preserving` | Replaces each character of the value with `X`, preserving length |
| `format_preserving_redact` | Replaces alphanumeric characters with `X`, preserving separators |
| `no_action` | Leaves the entity value unchanged |


## 1. Detect PII entities

We use `READIAnalyzer` with `DetectionType.PII_NO_MODEL`, which uses only regex and dictionary lookups — no spaCy or transformer models are loaded.

In [ ]:
from risk_assessment.readi.analyzer import READIAnalyzer

analyzer = READIAnalyzer(detection_type=READIAnalyzer.DetectionType.PII_NO_MODEL)

In [ ]:
text = """
Dear support team,

My name is Alice Johnson and I need help with my account.
You can reach me at alice.johnson@example.com or call me on +1-800-555-0199.
My SSN is 123-45-6789 and my credit card number is 4111 1111 1111 1111.
My IP address is 192.168.0.42 and my IBAN is GB29 NWBK 6016 1331 9268 19.

Best regards,
Alice
"""

entities = analyzer.detect(text)

print(f"Detected {len(entities)} entities:\n")
for entity in sorted(entities, key=lambda e: e.start):
    span = text[entity.start : entity.end]
    print(f"  [{entity.start}:{entity.end}]  {entity.entity_type:20s}  '{span}'")

## 2. Available masking actions

All masking actions share the same signature: `(entity_type: str, entity_text: str) -> str`.  
This makes it straightforward to compose them in a policy dictionary.

In [ ]:
from risk_assessment.masking.actions import (
    format_preserving_redact,
    no_action,
    redact_factory,
    redact_size_preserving,
    tagging_factory,
    tagging_with_hash,
)

# Show each action on a sample email address
sample_type = "Email"
sample_value = "alice.johnson@example.com"

print(f"Original value: '{sample_value}'\n")
print(f"  tagging_factory()       -> '{tagging_factory()(sample_type, sample_value)}'")
print(f"  tagging_with_hash       -> '{tagging_with_hash(sample_type, sample_value)}'")
print(f"  redact_factory()        -> '{redact_factory()(sample_type, sample_value)}'")
print(f"  redact_size_preserving  -> '{redact_size_preserving(sample_type, sample_value)}'")
print(f"  format_preserving_redact-> '{format_preserving_redact(sample_type, sample_value)}'")
print(f"  no_action               -> '{no_action(sample_type, sample_value)}'")

## 3. Apply a masking policy

A **masking policy** is a dictionary mapping entity types to masking actions.  
Types not listed in the policy fall back to a `default` action.

Here we apply the detected entities to the original text, processing spans from right to left so that earlier positions are not shifted by replacements.

In [ ]:
from collections.abc import Callable


def apply_masking(
    text: str,
    entities: list,
    policy: dict[str, Callable[[str, str], str]],
    default: Callable[[str, str], str] = tagging_factory(),
) -> str:
    """Apply masking actions to detected entities in a text.

    Processes entities from right to left to preserve character positions.

    Args:
        text: The original text.
        entities: List of Entity objects returned by READIAnalyzer.detect().
        policy: Mapping of entity_type -> masking action callable.
        default: Fallback action for types not listed in the policy.

    Returns:
        The masked text.
    """
    for entity in sorted(entities, key=lambda e: e.start, reverse=True):
        span = text[entity.start : entity.end]
        action = policy.get(entity.entity_type, default)
        replacement = action(entity.entity_type, span)
        text = text[: entity.start] + replacement + text[entity.end :]
    return text

In [ ]:
# Define the masking policy
# - Emails and phone numbers get a stable tag (EMAIL-1, PHONE-1, …)
# - Credit cards are format-preserving redacted (separators kept)
# - IBANs are fully redacted to a fixed placeholder
# - IP addresses are size-preserving redacted
# - Everything else (SSN, names, etc.) falls back to the default: tagging_with_hash

policy: dict[str, Callable[[str, str], str]] = {
    "Email": tagging_factory(),
    "Phone": tagging_factory(),
    "CreditCard": format_preserving_redact,
    "IBAN": redact_factory(symbol="*", size=5),
    "IP": redact_size_preserving,
}

masked_text = apply_masking(text, entities, policy=policy, default=tagging_with_hash)

print("=== Masked text ===")
print(masked_text)

## 4. Comparing masking strategies side by side

Let's run the same text through three different default strategies to compare the results.

In [ ]:
strategies = {
    "Stable tagging   ": tagging_factory(),
    "Hash tagging     ": tagging_with_hash,
    "Format preserving": format_preserving_redact,
}

for label, action in strategies.items():
    result = apply_masking(text, entities, policy={}, default=action)
    print(f"--- {label} ---")
    print(result)
    print()